# TP Elasticseach

L'objetif de ce TP est de découvrir les fonctionnalités d'Elasticsearch et de Kibana afin de construire un premier tableau de bord (_dashboard_) qui va permettre d'explorer un nouveau jeu de données.

Pour cela, il faut commencer par installer Elasticseach et Kibana :

```bash
docker pull nshou/elasticsearch-kibana
```

On peut maintenant lancer les containers Docker via la commande :

```bash
docker run -d -p 9200:9200 -p 5601:5601 nshou/elasticsearch-kibana
```

On désactive les warnings liés aux certificats SSL :

In [ ]:
import warnings
import urllib3

urllib3.disable_warnings()
warnings.filterwarnings('ignore')

Nous devons ensuite installer le package Python `elasticsearch` :

```bash
pip install "elasticsearch[async]>=8.0.0,<9.0.0"
```

In [ ]:
from IPython.display import JSON
from rich import print as rprint
from elasticsearch import Elasticsearch 
from elasticsearch import helpers
import pandas as pd

### Première connection 

Pour vous connecter vous devez trouver le mot de passe de votre container. Pour cela récupérez l'ID de votre container avec la commande `docker ps -a` puis afficher les logs avec `docker logs <container_id>`.

```bash
paul@LS-5CG4481WZ9:~$ docker ps -a
CONTAINER ID   IMAGE                        COMMAND                CREATED         STATUS         PORTS                                                                                      NAMES
60e759c92d4f   nshou/elasticsearch-kibana   "bash entrypoint.sh"   5 minutes ago   Up 5 minutes   0.0.0.0:5601->5601/tcp, [::]:5601->5601/tcp, 0.0.0.0:9200->9200/tcp, [::]:9200->9200/tcp   modest_archimedes
```

L'ID du container est ici `60e759c92d4f`. On peut donc regarder dans les logs :

```bash
paul@pc-scalian:~$ docker logs 60e759c92d4f
----------------------------------------------------------------------------------
------------------------- STARTING ELASTIC/KIBANA --------------------------------
SSL ENABLED: 'true'
----------------------------------------------------------------------------------
Kibana is currently running with legacy OpenSSL providers enabled! For details and instructions on how to disable see https://www.elastic.co/guide/en/kibana/8.10/production.html#openssl-legacy-provider
{"log.level":"info","@timestamp":"2026-03-27T08:06:12.974Z","log":{"logger":"elastic-apm-node"},"agentVersion":"3.49.1","env":{"pid":151,"proctitle":"kibana-8.10.4/bin/../node/bin/node","os":"linux 6.6.87.2-microsoft-standard-WSL2","arch":"x64","host":"60e759c92d4f","timezone":"UTC+00","runtime":"Node.js v18.17.1"},"config":{"serviceName":{"source":"start","value":"kibana","commonName":"service_name"},"serviceVersion":{"source":"start","value":"8.10.4","commonName":"service_version"},"serverUrl":{"source":"start","value":"https://kibana-cloud-apm.apm.us-east-1.aws.found.io/","commonName":"server_url"},"logLevel":{"source":"default","value":"info","commonName":"log_level"},"active":{"source":"start","value":true},"contextPropagationOnly":{"source":"start","value":true},"environment":{"source":"start","value":"production"},"logUncaughtExceptions":{"source":"start","value":true},"globalLabels":{"source":"start","value":[["git_rev","976088dd04c6fd3b907fd2bb92af306e7d77ce4c"]],"sourceValue":{"git_rev":"976088dd04c6fd3b907fd2bb92af306e7d77ce4c"}},"secretToken":{"source":"start","value":"[REDACTED]","commonName":"secret_token"},"breakdownMetrics":{"source":"start","value":false},"captureSpanStackTraces":{"source":"start","sourceValue":false},"centralConfig":{"source":"start","value":false},"metricsInterval":{"source":"start","value":120,"sourceValue":"120s"},"propagateTracestate":{"source":"start","value":true},"transactionSampleRate":{"source":"start","value":0.1,"commonName":"transaction_sample_rate"},"captureBody":{"source":"start","value":"off","commonName":"capture_body"},"captureHeaders":{"source":"start","value":false}},"activationMethod":"require","ecs":{"version":"1.6.0"},"message":"Elastic APM Node.js Agent v3.49.1"}
Mar 27, 2026 8:06:13 AM sun.util.locale.provider.LocaleProviderAdapter <clinit>
WARNING: COMPAT locale provider will be removed in a future release
----------------------------------------------------------------------------------
Elasticsearch + Kibana is now fully configured, you may access the stack below
     URL: https://localhost:5601/
    User: 'elastic'
Password: 'ugmWZ_2-1MtMApQDGUpe'
----------------------------------------------------------------------------------
```

Le mot de passe à reporter est donc dans cet exemple `ugmWZ_2-1MtMApQDGUpe`.

In [ ]:
PASSWORD = 'ae02n8i1AWK*iqfTVW=-'

es_client = Elasticsearch(
    hosts=["https://localhost:9200"],  # Utilisation directe de l'URL avec https
    basic_auth=("elastic", PASSWORD),  # Utilisateur et mot de passe
    http_compress=True,
    verify_certs=False
)
response = es_client.info()
JSON(dict(response))

### Initialisation et création de l'index

Téléchargez ces données et importez les dans un dataframe avec pandas
https://www.kaggle.com/roshansharma/sanfranciso-crime-dataset?select=Police_Department_Incidents_-_Previous_Year__2016_.csv

In [ ]:
# TODO lire le csv à l'aide de pandas
df = None

# on formate la localisation pour la suite du TP
df['Location'] = df['Location'].astype(str).str.replace(r'[\(\)\s]', '', regex=True)

In [ ]:
df.head(2)

### Création de l'index vide

In [ ]:
from elasticsearch import Elasticsearch
from elasticsearch import helpers

INDEX = "sf_crimes"

# TODO créer un index (vide pour l'instant) "sf_crimes" dans lequel on rajoutera les données
# utiliser indices.create
if es_client.indices.exists(index=INDEX):
    es_client.indices.delete(index=INDEX)

response = es_client.indices.create(index=INDEX, body={})
rprint(dict(response))

On visualise les paramètres de l'index :

In [ ]:
#TODO

On peut également afficher son mapping :

In [ ]:
#TODO

Comme attendu, celui-ci est bien vide.

### Premièrs insertions

À l'aide de l'API `helpers.bulk`, insérez tout le DataFrame dans l'index que l'on vient de créer.

In [ ]:
def get_data(df, index_name):
    for i, row in df.iterrows():
        yield {
            "_index": index_name,
            "_id": str(i),
            "_source": row.to_dict()
        }

# Envoi groupé
helpers.bulk(es_client, get_data(df, INDEX))

Cela plante car nous n'avions pas géré les valeurs `NaN` :

In [ ]:
df.isna().sum()

In [ ]:
import numpy as np

df_cleaned = df.replace({np.nan: None}) 

Comme on a inséré à partir de l'index, nous ne sommes pas obligés de tout supprimer. Le Bulk va réécrire et donc ne pas modifier ce qui existe déjà en base.

In [ ]:
try:
    helpers.bulk(es_client, get_data(df_cleaned, INDEX))
    print("Données insérées avec succès.")
except helpers.BulkIndexError as e:
    print(f"Erreur lors de l'insertion des données : {e}")
    for error in e.errors:
        print("Erreur spécifique :", error)

In [ ]:
helpers.bulk(es_client, get_data(df_cleaned.head(10), INDEX))

On vérifie l'insertion :

In [ ]:
# on vide les tampons mémoire et on rendre les documents récemment insérés disponibles pour la recherche
es_client.indices.refresh(index=INDEX)
# connaitre le volume (nombre de documents) dans l'index
es_client.cat.count(index=INDEX, params={"format": "json"})

On peut voir que 150 500 documents ont été indexés, soit le nombre de lignes du DataFrame :

In [ ]:
df_cleaned.shape[0]

On affiche le mapping :

In [ ]:
es_client.indices.refresh(index=INDEX)
es_client.indices.get_mapping(index=INDEX)

On peut s'apercevoir que le mapping par défaut n'est pas judicieux :
* La propriété `Date` n'est pas traîtée comme telle.
* Il existe pour la propriété `Localisation` un type `geo_point` pour les coordonnées géographiques.

Il va donc nous falloir créer un mapping et réinsérer les données.

### On va ajouter un mapping

On commence par supprimer l'index.

In [ ]:
es_client.indices.delete(index=INDEX)

In [ ]:
df.head(1)

In [ ]:
df.dtypes

Voir la documentation officielle :
* pour les dates : https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/mapping-date-format
* pour les `geo_point` : https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/geo-point

In [ ]:
# TODO Créez un dictionnaire Python contenant le mapping
# format date : MM/dd/yyyy hh:mm:ss a
# type pour la propriété Localisation : geo_point
mapping = {}

In [ ]:
df_cleaned

In [ ]:
# On commence par supprimer l'index précédent (s'il existe bien)
if es_client.indices.exists(index=INDEX):
    es_client.indices.delete(index=INDEX)

# Créer un nouvel index avec le mapping corrigé
es_client.indices.create(index=INDEX, body=mapping)

In [ ]:
try:
    helpers.bulk(es_client, get_data(df_cleaned, INDEX))
    print("Données insérées avec succès.")
except helpers.BulkIndexError as e:
    print(f"Erreur lors de l'insertion des données : {e}")
    for error in e.errors:
        print("Erreur spécifique :", error)

In [ ]:
df_cleaned.head(2)

In [ ]:
es_client.indices.refresh(index=INDEX)
es_client.indices.get_mapping(index=INDEX)

### Kibana 

Maintenant que les données sont bien formées, nous pouvons les visualiser dans Kibana. Pour cela, allez sur https://localhost:5601/

#### Création d'une première visualisation 

Maintenant vous allez pouvoir commencer à créer plusieurs visualisations que vous rassemblerez dans un dashboard. Commencez par créer une carte qui géolocalise les crimes.

#### Création d'un _dashboard_

Créez un _dashboard_ puis ajoutez-y la carte des crimes. Créez ensuite de nouvelles visualisations.